In [1]:
import pandas as pd
import missingno as msno
import pandas as pd
import numpy as np
import re
import csv
import os

# local src

from src.clean_column_names import clean_column_names
from src.df_overview import df_overview
from src.download_data import download_data

In [2]:
download_data(dataset_path="arashnic/book-recommendation-dataset")

Data exists in D:\projects\sql-books\data\01_raw. Skipping.


In [3]:
df = pd.read_csv("../data/01_raw/Books.csv")

C:\Users\zxrco\AppData\Local\Temp\ipykernel_9640\3809504026.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/01_raw/Books.csv")


In [4]:
import pandas as pd
import numpy as np
import re
import os

# 0. CONFIG
os.makedirs("../data/02_quarantine", exist_ok=True)
os.makedirs("../data/02_processed", exist_ok=True)

CURRENT_YEAR = pd.Timestamp.now().year
ISBN_REGEX   = r"[\dX\-]{8,13}"
ASIN_REGEX   = r"B[\dA-Z]{9}"


# 1. LOAD
def load_raw(path):
    return pd.read_csv(
        path,
        encoding="latin-1",
        dtype=str,
        keep_default_na=False,
        low_memory=False
    )

books   = load_raw("../data/01_raw/Books.csv")
users   = load_raw("../data/01_raw/Users.csv")
ratings = load_raw("../data/01_raw/Ratings.csv")

for df in [books, users, ratings]:
    df.columns = df.columns.str.strip()


# 2. STANDARDIZE
def standardize_books(df):
    df = df.copy()
    df[df.select_dtypes("object").columns] = (
        df.select_dtypes("object").apply(lambda c: c.str.strip())
    )
    df["Book-Author"] = df["Book-Author"].str.title()
    df["Publisher"]   = df["Publisher"].str.title()
    df["ISBN"]        = df["ISBN"].str.upper()
    df["Year-Of-Publication"] = (
        pd.to_numeric(df["Year-Of-Publication"], errors="coerce").astype("Int64")
    )
    return df

def standardize_users(df):
    df = df.copy()
    df["Location"] = df["Location"].str.strip().str.lower()
    df["User-ID"]  = pd.to_numeric(df["User-ID"], errors="coerce").astype("Int64")
    df["Age"]      = pd.to_numeric(df["Age"],     errors="coerce").astype("Int64")
    return df

def standardize_ratings(df):
    df = df.copy()
    df["ISBN"]        = df["ISBN"].str.strip().str.upper()
    df["User-ID"]     = pd.to_numeric(df["User-ID"],     errors="coerce").astype("Int64")
    df["Book-Rating"] = pd.to_numeric(df["Book-Rating"], errors="coerce").astype("Int64")
    return df

books   = standardize_books(books)
users   = standardize_users(users)
ratings = standardize_ratings(ratings)


# 3. SCHEMA  (config-driven)
def _make_schema():

    def mask(series, fn):
        """Apply fn safely, treating NA as not-bad."""
        return fn(series)

    books_rules = [
        {
            "col":     "ISBN",
            "mask_fn": lambda s: s == "",
            "message": "ISBN is empty",
        },
        {
            "col":     "ISBN",
            "mask_fn": lambda s: s.str.fullmatch(ASIN_REGEX, na=False),
            "message": lambda s: "ASIN not ISBN: " + s,
        },
        {
            "col":     "ISBN",
            "mask_fn": lambda s: ~s.str.fullmatch(ISBN_REGEX, na=False) & (s != ""),
            "message": lambda s: "Malformed ISBN: " + s,
        },
        {
            "col":     "Book-Title",
            "mask_fn": lambda s: s == "",
            "message": "Book-Title is empty",
        },
        {
            "col":     "Book-Title",
            "mask_fn": lambda s: s.str.fullmatch(ISBN_REGEX, na=False),
            "message": lambda s: "Book-Title looks like an ISBN: " + s,
        },
        {
            "col":     "Book-Author",
            "mask_fn": lambda s: s == "",
            "message": "Book-Author is empty",
        },
        {
            "col":     "Book-Author",
            "mask_fn": lambda s: s.str.len() <= 1,
            "message": lambda s: "Book-Author too short: " + s,
        },
        {
            "col":     "Book-Author",
            "mask_fn": lambda s: s.str.fullmatch(r"\d{4}", na=False),
            "message": lambda s: "Book-Author looks like a year: " + s,
        },
        {
            "col":     "Year-Of-Publication",
            "mask_fn": lambda s: s.notna() & ~s.between(1450, CURRENT_YEAR),
            "message": lambda s: "Year out of range: " + s.astype(str),
        },
        {
            "col":     "Image-URL-S",
            "mask_fn": lambda s: (s != "") & ~s.str.startswith("http"),
            "message": lambda s: "Image-URL-S not a URL: " + s.str[:40],
        },
        {
            "col":     "Image-URL-M",
            "mask_fn": lambda s: (s != "") & ~s.str.startswith("http"),
            "message": lambda s: "Image-URL-M not a URL: " + s.str[:40],
        },
        {
            "col":     "Image-URL-L",
            "mask_fn": lambda s: (s != "") & ~s.str.startswith("http"),
            "message": lambda s: "Image-URL-L not a URL: " + s.str[:40],
        },
    ]

    users_rules = [
        {
            "col":     "Age",
            "mask_fn": lambda s: s.notna() & ~s.between(0, 120),
            "message": lambda s: "Age out of range: " + s.astype(str),
        },
    ]

    ratings_rules = [
        {
            "col":     "Book-Rating",
            "mask_fn": lambda s: s.isna(),
            "message": "Rating is null",
        },
        {
            "col":     "Book-Rating",
            "mask_fn": lambda s: s.notna() & ~s.between(0, 10),
            "message": lambda s: "Rating out of range: " + s.astype(str),
        },
    ]

    return {
        "books":   books_rules,
        "users":   users_rules,
        "ratings": ratings_rules,
    }

SCHEMA = _make_schema()


# 4. VALIDATE
def apply_schema(df, rules):
    """
    For each rule, compute bad-row mask and resolve message.
    Returns a Series of combined reason strings (empty = clean).
    """
    reason_parts = []

    for rule in rules:
        col     = rule["col"]
        mask_fn = rule["mask_fn"]
        message = rule["message"]

        if col not in df.columns:
            continue

        bad_mask = mask_fn(df[col])

        if callable(message):
            reasons = pd.Series("", index=df.index)
            reasons[bad_mask] = message(df.loc[bad_mask, col])
        else:
            reasons = pd.Series(
                np.where(bad_mask, message, ""), index=df.index
            )

        reason_parts.append(reasons)

    if not reason_parts:
        return pd.Series("", index=df.index)

    reason_df = pd.concat(reason_parts, axis=1)
    reason_df.columns = range(len(reason_df.columns))

    return (
        reason_df.stack()
                 .loc[lambda s: s != ""]
                 .groupby(level=0)
                 .agg(" | ".join)
                 .reindex(df.index, fill_value="")
    )


def validate_and_split(df, rules):
    combined = apply_schema(df, rules)
    bad_mask = combined != ""
    good_df  = df[~bad_mask].reset_index(drop=True)
    bad_df   = df[bad_mask].copy()
    bad_df["reason"] = combined[bad_mask].values
    return good_df, bad_df.reset_index(drop=True)


books_clean,   books_bad   = validate_and_split(books,   SCHEMA["books"])
users_clean,   users_bad   = validate_and_split(users,   SCHEMA["users"])
ratings_clean, ratings_bad = validate_and_split(ratings, SCHEMA["ratings"])


# 5. REFERENTIAL INTEGRITY (the same as FK checks in SQL)
orphan_isbn_mask  = ~ratings_clean["ISBN"].isin(books_clean["ISBN"])
orphan_user_mask  = ~ratings_clean["User-ID"].isin(users_clean["User-ID"])

orphan_isbn           = ratings_clean[orphan_isbn_mask].copy()
orphan_isbn["reason"] = "ISBN not found in books"

orphan_user           = ratings_clean[orphan_user_mask & ~orphan_isbn_mask].copy()
orphan_user["reason"] = "User-ID not found in users"

ratings_clean  = ratings_clean[~orphan_isbn_mask & ~orphan_user_mask].reset_index(drop=True)
ratings_bad    = pd.concat([ratings_bad, orphan_isbn, orphan_user], ignore_index=True)


# 6. DUPLICATES
def drop_dupes_to_quarantine(df, subset, bad_df, reason):
    mask            = df.duplicated(subset=subset, keep="first")
    dupes           = df[mask].copy()
    dupes["reason"] = reason
    return (
        df[~mask].reset_index(drop=True),
        pd.concat([bad_df, dupes], ignore_index=True)
    )

books_clean,   books_bad   = drop_dupes_to_quarantine(books_clean,   ["ISBN"],            books_bad,   "Duplicate ISBN")
users_clean,   users_bad   = drop_dupes_to_quarantine(users_clean,   ["User-ID"],         users_bad,   "Duplicate User-ID")
ratings_clean, ratings_bad = drop_dupes_to_quarantine(ratings_clean, ["User-ID", "ISBN"], ratings_bad, "Duplicate (User-ID, ISBN)")


# 7. EXPORT
books_bad.to_csv("../data/02_quarantine/books_quarantine.csv",     index=False)
users_bad.to_csv("../data/02_quarantine/users_quarantine.csv",     index=False)
ratings_bad.to_csv("../data/02_quarantine/ratings_quarantine.csv", index=False)

books_clean.to_csv("../data/02_processed/books_good.csv",          index=False)
users_clean.to_csv("../data/02_processed/users_good.csv",          index=False)
ratings_clean.to_csv("../data/02_processed/ratings_good.csv",      index=False)


# 8. SUMMARY
print("\n" + "=" * 50)
print("QUALITY SUMMARY")
print("=" * 50)

for name, clean, bad in [
    ("Books",   books_clean,   books_bad),
    ("Users",   users_clean,   users_bad),
    ("Ratings", ratings_clean, ratings_bad),
]:
    total = len(clean) + len(bad)
    print(f"\n{name}:")
    print(f"  Total raw:   {total:>8,}")
    print(f"  Clean:       {len(clean):>8,}  ({len(clean)/total*100:.1f}%)")
    print(f"  Quarantined: {len(bad):>8,}  ({len(bad)/total*100:.1f}%)")
    if len(bad) > 0:
        print("  Top reasons:")
        print(bad["reason"].value_counts().head(5).to_string(header=False))


QUALITY SUMMARY

Books:
  Total raw:    271,360
  Clean:        266,345  (98.2%)
  Quarantined:    5,015  (1.8%)
  Top reasons:
Year out of range: 0                               4558
Duplicate ISBN                                      308
Year out of range: 2030                               7
Book-Author too short: X | Year out of range: 0       5
Book-Author too short: N                              4

Users:
  Total raw:    278,858
  Clean:        278,780  (100.0%)
  Quarantined:       78  (0.0%)
  Top reasons:
Age out of range: 123    8
Age out of range: 148    5
Age out of range: 124    5
Age out of range: 204    4
Age out of range: 210    3

Ratings:
  Total raw:   1,149,780
  Clean:       1,015,862  (88.4%)
  Quarantined:  133,918  (11.6%)
  Top reasons:
ISBN not found in books       132823
User-ID not found in users      1095
